In [1]:
from dotenv import load_dotenv
import os
import requests
from datetime import datetime
from pprint import pprint

load_dotenv('apis.env')
API_KEY = os.getenv('API')

#### API 1: NEOWS


In [7]:
class Asteroid:
    def __init__(self, name, diameter, velocity, miss_distance):
        self.name = name
        self.diameter = diameter
        self.velocity = velocity
        self.miss_distance = miss_distance
        self.is_hazardous = False
        self.is_sentry_object = False
        self.orbital_data = None

    def __repr__(self):
        return f"Asteroid(name={self.name}, diameter={self.diameter}, velocity={self.velocity}, miss_distance={self.miss_distance})"

In [3]:
params = {
    'start_date' : '2026-07-16',
    'end_date'   : '2026-07-16',
    'api_key'    : API_KEY
}
results = requests.get("https://api.nasa.gov/neo/rest/v1/feed", params=params)

In [5]:
# print(params)
# print(results.json()[0].keys())
data = results.json()
no_of_asteroids:int = int(data['element_count'])
print(f"no:asteroids : {no_of_asteroids}\n\n\n\n")
# pprint.pprint(data)
pprint(data['near_earth_objects'])

no:asteroids : 5




{'2026-07-16': [{'absolute_magnitude_h': 23.05,
                 'close_approach_data': [{'close_approach_date': '2026-07-16',
                                          'close_approach_date_full': '2026-Jul-16 '
                                                                      '06:48',
                                          'epoch_date_close_approach': 1784184480000,
                                          'miss_distance': {'astronomical': '0.4620301279',
                                                            'kilometers': '69118723.009667573',
                                                            'lunar': '179.7297197531',
                                                            'miles': '42948382.9352275874'},
                                          'orbiting_body': 'Earth',
                                          'relative_velocity': {'kilometers_per_hour': '95584.8664596629',
                                                           

In [ ]:
todays_asteroids_data = []
for asteroid_data in data['near_earth_objects']['2026-07-16']:

    asteroid = {
        'name': asteroid_data['name'],
        'id': asteroid_data['id'],
        'diameter_min': asteroid_data['estimated_diameter']['meters']['estimated_diameter_min'],
        'diameter_max': asteroid_data['estimated_diameter']['meters']['estimated_diameter_max'],
        'is_hazardous': asteroid_data['is_potentially_hazardous_asteroid'],
        'is_sentry_object': asteroid_data['is_sentry_object'],
        'miss_distance': asteroid_data['close_approach_data'][0]['miss_distance']['kilometers']
    }
    todays_asteroids_data.append(asteroid)

print(f"todays_asteroids_data : {todays_asteroids_data}\n\n\n\n")

todays_asteroids_data : [{'name': '(2009 DB1)', 'id': '3447916', 'diameter_min': 65.2461629789, 'diameter_max': 145.8948556919, 'is_hazardous': False, 'is_sentry_object': False}, {'name': '(2009 HA21)', 'id': '3457840', 'diameter_min': 179.7028548592, 'diameter_max': 401.8277992159, 'is_hazardous': True, 'is_sentry_object': False}, {'name': '(2017 FX101)', 'id': '3772995', 'diameter_min': 22.3128464417, 'diameter_max': 49.8930414151, 'is_hazardous': False, 'is_sentry_object': False}, {'name': '(2018 XQ2)', 'id': '3836857', 'diameter_min': 6.0891262211, 'diameter_max': 13.6157001539, 'is_hazardous': False, 'is_sentry_object': True}, {'name': '(2019 NX4)', 'id': '3843209', 'diameter_min': 160.1603379786, 'diameter_max': 358.1294030194, 'is_hazardous': False, 'is_sentry_object': False}]






### API 2 SDBD database


In [12]:
SDBD_API = f"https://ssd-api.jpl.nasa.gov/sbdb.api"

def get_orbit_value(elements, target_name):
    for element in elements:
        if element.get('name') == target_name:
            value = element.get('value')
            if value is None:
                return None
            return float(value)
    return None

for asteroid_data in todays_asteroids_data:
    response = requests.get(SDBD_API, params={'spk': asteroid_data['id']})
    response.raise_for_status()  # Raise an exception for HTTP errors

    data2 = response.json()
    # pprint.pprint(data2)

    # object data
    asteroid_data['spk_id'] = int(data2['object']['spkid'])
    asteroid_data['des'] = data2['object']['des']
    asteroid_data['neo'] = data2['object']['neo']
    asteroid_data['pha'] = data2['object']['pha']
    asteroid_data['orbit_class_name'] = data2['object']['orbit_class']['name']

    # orbit data
    asteroid_data['semi-major-axis'] = get_orbit_value(data2['orbit']['elements'], 'a')
    asteroid_data['eccentricity'] = get_orbit_value(data2['orbit']['elements'], 'e')
    asteroid_data['inclination'] = get_orbit_value(data2['orbit']['elements'], 'i')
    asteroid_data['longitude_of_ascending_node'] = get_orbit_value(data2['orbit']['elements'], 'om')
    asteroid_data['argument_of_perihelion'] = get_orbit_value(data2['orbit']['elements'], 'w')
    asteroid_data['mean_anomaly'] = get_orbit_value(data2['orbit']['elements'], 'ma')
    asteroid_data['orbital_period'] = get_orbit_value(data2['orbit']['elements'], 'per')
    asteroid_data['condition_code'] = int(data2['orbit']['condition_code'])
    asteroid_data['first_observation_date'] = data2['orbit']['first_obs']
    asteroid_data['soln_date'] = data2['orbit']['soln_date']
    asteroid_data['rms'] = float(data2['orbit']['rms'])
    asteroid_data['moid'] = float(data2['orbit']['moid'])
    asteroid_data['data_arc'] = int(data2['orbit']['data_arc'])
    asteroid_data['n_obs_used'] = data2['orbit']['n_obs_used']

#printing the final data for all asteroids
pprint(todays_asteroids_data)

[{'argument_of_perihelion': 240.0,
  'condition_code': 0,
  'data_arc': 5493,
  'des': '2009 DB1',
  'diameter_max': 145.8948556919,
  'diameter_min': 65.2461629789,
  'eccentricity': 0.44,
  'first_observation_date': '2009-02-19',
  'id': '3447916',
  'inclination': 4.44,
  'is_hazardous': False,
  'is_sentry_object': False,
  'longitude_of_ascending_node': 168.0,
  'mean_anomaly': 293.0,
  'moid': 0.0387,
  'n_obs_used': 78,
  'name': '(2009 DB1)',
  'neo': True,
  'orbit_class_name': 'Apollo',
  'orbital_period': 499.0,
  'pha': False,
  'rms': 0.57,
  'semi-major-axis': 1.23,
  'soln_date': '2024-03-05 05:00:03',
  'spk_id': 50447939},
 {'argument_of_perihelion': 220.0,
  'condition_code': 1,
  'data_arc': 5109,
  'des': '2009 HA21',
  'diameter_max': 401.8277992159,
  'diameter_min': 179.7028548592,
  'eccentricity': 0.729,
  'first_observation_date': '2009-04-20',
  'id': '3457840',
  'inclination': 6.41,
  'is_hazardous': True,
  'is_sentry_object': False,
  'longitude_of_ascend

### API 3 JPL Horizons API

In [13]:
import re
import requests
from pprint import pprint

JPL_API = "https://ssd.jpl.nasa.gov/api/horizons.api"


def extract_horizons_vectors(result_text):
    """
    Simple parser for Horizons VECTORS output.
    Finds data between $$SOE and $$EOE markers.
    """
    vectors = []
    lines = result_text.splitlines()

    # Find the SOE and EOE markers
    soe_idx = None
    eoe_idx = None
    for i, line in enumerate(lines):
        if line.strip() == '$$SOE':
            soe_idx = i
        elif line.strip() == '$$EOE':
            eoe_idx = i
            break

    if soe_idx is None or eoe_idx is None:
        print("Warning: Could not find $$SOE or $$EOE in response")
        return vectors

    # Get only the lines between the markers
    data_lines = lines[soe_idx + 1 : eoe_idx]

    # This pattern finds numbers like 5.195E+07 or -1.734E+01
    number_pattern = r'[-+]?\d+\.\d+E[+-]\d+'

    i = 0
    while i < len(data_lines):
        line = data_lines[i].strip()

        # Skip empty lines
        if not line:
            i += 1
            continue

        # Time lines always contain "A.D."
        if 'A.D.' in line:
            # Extract Julian Day: everything before the first "="
            jd = float(line.split('=')[0].strip())

            # Extract datetime: between "A.D." and "TDB"
            datetime_str = line.split('A.D.')[1].split('TDB')[0].strip()

            # Next line: Position (X, Y, Z)
            i += 1
            if i >= len(data_lines):
                break
            pos_line = data_lines[i]
            pos_numbers = re.findall(number_pattern, pos_line)

            if len(pos_numbers) != 3:
                print(f"Warning: Expected 3 position numbers, got {len(pos_numbers)}")
                break

            x, y, z = map(float, pos_numbers)

            # Next line: Velocity (VX, VY, VZ)
            i += 1
            if i >= len(data_lines):
                break
            vel_line = data_lines[i]
            vel_numbers = re.findall(number_pattern, vel_line)

            if len(vel_numbers) != 3:
                print(f"Warning: Expected 3 velocity numbers, got {len(vel_numbers)}")
                break

            vx, vy, vz = map(float, vel_numbers)

            # Store this data point
            vectors.append({
                'jd': jd,
                'datetime': datetime_str,
                'x_km': x,
                'y_km': y,
                'z_km': z,
                'vx_kms': vx,
                'vy_kms': vy,
                'vz_kms': vz,
            })

        i += 1

    return vectors


# Your loop
for asteroid_data in todays_asteroids_data:
    params = {
        'format': 'json',
        'EPHEM_TYPE': 'VECTORS',
        'COMMAND': f"'DES={asteroid_data['spk_id']};'",
        'CENTER': '500@399',
        'START_TIME': "'2026-07-16 00:00'",
        'STOP_TIME': "'2026-07-16 02:00'",
        'STEP_SIZE': '120m',
        'REF_PLANE': 'ECLIPTIC',
        'REF_SYSTEM': 'ICRF',
        'OUT_UNITS': 'KM-S',
        'VEC_TABLE': '2',
        'TIME_TYPE': 'TDB',
        'TIME_DIGITS': 'SECONDS',
    }

    response3 = requests.get(JPL_API, params=params)
    response3.raise_for_status()
    data3 = response3.json()
    vectors = extract_horizons_vectors(data3['result'])

    pprint({
        'name': asteroid_data['name'],
        'spk_id': asteroid_data['spk_id'],
        'vectors': vectors,
    })

    # adding the vectors to the asteroid data
    asteroid_data['vectors'] = vectors

{'name': '(2009 DB1)',
 'spk_id': 50447939,
 'vectors': [{'datetime': '2026-Jul-16 00:00:00.0000',
              'jd': 2461237.5,
              'vx_kms': -17.34485556901394,
              'vy_kms': 19.91990971331239,
              'vz_kms': -2.547749725956407,
              'x_km': 51952073.46120522,
              'y_km': 45281142.71769203,
              'z_km': 5320527.640874883},
             {'datetime': '2026-Jul-16 02:00:00.0000',
              'jd': 2461237.583333333,
              'vx_kms': -17.3624325541164,
              'vy_kms': 19.91049251609708,
              'vz_kms': -2.549373568723116,
              'x_km': 51827127.24229449,
              'y_km': 45424532.15778583,
              'z_km': 5302177.995243879}]}
{'name': '(2009 HA21)',
 'spk_id': 50457863,
 'vectors': [{'datetime': '2026-Jul-16 00:00:00.0000',
              'jd': 2461237.5,
              'vx_kms': -24.23847331844829,
              'vy_kms': 23.61534513566672,
              'vz_kms': -3.443280606612897,
    

In [ ]:
import math
# add distance from earth to the asteroid data
for asteroid_data in todays_asteroids_data:
    distance_from_earth = math.sqrt(asteroid_data['x_km']**2 + asteroid_data['y_km']**2 + asteroid_data['z_km']**2)
    asteroid_data['distance_from_earth'] = distance_from_earth

In [17]:
# checking close approach data
if not todays_asteroids_data:
    raise ValueError("No asteroid data available to query close approach data.")

params = {
    'format': 'json',
    'EPHEM_TYPE': 'APPROACH',
    'COMMAND': f"'DES={todays_asteroids_data[0]['spk_id']};'",
    'CENTER': '500@399',  # Earth
    'START_TIME': "'2026-07-16'",
    'STOP_TIME': "'2026-07-18'",
}
response4 = requests.get(JPL_API, params=params)
response4.raise_for_status()

data4 = response4.json()
pprint(data4)

{'result': '*******************************************************************************\n'
           'JPL/HORIZONS                     (2009 DB1)                '
           '2026-Jul-18 11:33:30\n'
           'Rec #:50044718 (+COV) Soln.date: 2024-Mar-05_05:00:03     # obs: '
           '78 (2009-2024)\n'
           ' \n'
           'IAU76/J2000 helio. ecliptic osc. elements (au, days, deg., '
           'period=Julian yrs):\n'
           ' \n'
           '  EPOCH=  2456704.5 ! 2014-Feb-16.00 (TDB)         Residual RMS= '
           '.26368\n'
           '   EC= .4396708061734013   QR= .689156852165505    TP= '
           '2456805.6758037037\n'
           '   OM= 168.0625234718097   W=  239.8181426317114   IN= '
           '4.423219967870643\n'
           '   A= 1.229914235699762    MA= 286.8913954581936   ADIST= '
           '1.77067161923402\n'
           '   PER= 1.36402            N= .7225898079999999    ANGMOM= '
           '.017134525\n'
           '   DAN= 1.2737          